<a href="https://colab.research.google.com/github/Anthei0774/Advent-of-Code/blob/main/2023/Day_23_A_Long_Walk.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
puzzle = '''#.#####################
#.......#########...###
#######.#########.#.###
###.....#.>.>.###.#.###
###v#####.#v#.###.#.###
###.>...#.#.#.....#...#
###v###.#.#.#########.#
###...#.#.#.......#...#
#####.#.#.#######.#.###
#.....#.#.#.......#...#
#.#####.#.#.#########v#
#.#...#...#...###...>.#
#.#.#v#######v###.###v#
#...#.>.#...>.>.#.###.#
#####v#.#.###v#.#.###.#
#.....#...#...#.#.#...#
#.#########.###.#.#.###
#...###...#...#...#.###
###.###.#.###v#####v###
#...#...#.#.>.>.#.>.###
#.###.###.#.###.#.#v###
#.....###...###...#...#
#####################.#'''

with open('23.txt') as f: puzzle = f.read()

puzzle = puzzle.split('\n')
puzzle = [list(line) for line in puzzle]
puzzle

###############################################################################
# PART 1

# Suggested by https://www.reddit.com/r/adventofcode/comments/18oy4pc/comment/kel78ae/?utm_source=share&utm_medium=web2x&context=3

#
R = len(puzzle)
C = len(puzzle[0])
R, C

# directed graph
graph = {}
for r in range(R):
    for c in range(C):
        char = puzzle[r][c]
        if char != '#':
            neighbours = None
            if char == '.':
                neighbours = [(r - 1, c), (r, c - 1), (r, c + 1), (r + 1, c)]
                neighbours = [n for n in neighbours if 0 <= n[0] and n[0] < R and 0 <= n[1] and n[1] < C]
                neighbours = [n for n in neighbours if puzzle[n[0]][n[1]] != '#']
            else:
                if char == '>': neighbours = [(r, c + 1)]
                elif char == '<': neighbours = [(r, c - 1)]
                elif char == 'v': neighbours = [(r + 1, c)]
                else: neighbours = [(r - 1, c)]
            graph[(r, c)] = neighbours
graph

#
start = (0, puzzle[0].index('.'))
end = (R - 1, puzzle[R - 1].index('.'))
start, end

def calculate_crossroads():
    global graph, start, end

    # create crossroads skeleton containing intersections
    crossroads = {node: {} for node in graph if len(graph[node]) in [3, 4]}

    # add start-end
    crossroads[start] = {}
    crossroads[end] = {}

    # for each crossroad...
    for cr in crossroads:

        # take their immediate neighbours and start paths simultaneously
        paths = graph[cr]
        paths = [[cr, p] for p in paths]

        # in each iter, expand the paths if possible
        while True:

            next = []
            for p in paths:
                neighbours = graph[p[-1]]
                for n in neighbours:
                    if n not in p:
                        new = p + [n]

                        # if expanded path end is another intersection, then finish this one, marking the distance
                        if n in crossroads:
                            crossroads[cr][n] = len(new) - 1
                        else:
                            next.append(new)
            if next != []:
                paths = next
            else:
                break

    return crossroads

crossroads = calculate_crossroads()
crossroads

def calculate_longest_hike():
    global start, end, crossroads

    current = [[start]]
    D = 0
    while True:

        next = []
        for path in current:

            #
            neighbours = crossroads[path[-1]]
            for n in neighbours:
                if n not in path:
                    new = path + [n]
                    if n != end:
                        next.append(new)
                    else:
                        d = sum(crossroads[new[i - 1]][new[i]] for i in range(1, len(new)))
                        if D < d:
                            D = d
                            print(D, new)

        if next != []:
            current = next
        else:
            break

calculate_longest_hike()

###############################################################################
# PART 2

# rewrite graph into undirected
graph = {}
for r in range(R):
    for c in range(C):
        char = puzzle[r][c]
        if char != '#':
            neighbours = None
            neighbours = [(r - 1, c), (r, c - 1), (r, c + 1), (r + 1, c)]
            neighbours = [n for n in neighbours if 0 <= n[0] and n[0] < R and 0 <= n[1] and n[1] < C]
            neighbours = [n for n in neighbours if puzzle[n[0]][n[1]] != '#']
            graph[(r, c)] = neighbours
graph

# recalculate crossroads
crossroads = calculate_crossroads()
calculate_longest_hike()